<a href="https://colab.research.google.com/github/Pruthviraj141/Disease-and-symptoms-dataset-Training/blob/main/Disease_and_symptoms_dataset_Training_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip /content/d.zip -d /content/


Archive:  /content/d.zip
  inflating: /content/Disease and symptoms dataset 2023/Disease and symptoms dataset.csv  


In [3]:
"""
Disease Prediction from Symptoms — GPU-Accelerated Multi-Model Pipeline
==============================================================================
Optimized for Google Colab (T4 GPU).
Limits system RAM usage by converting data to float32 and natively routes
Scikit-Learn workloads to the GPU via NVIDIA cuML. Outputs saved .pkl models.
"""

import os
import argparse
import sys
import time
import warnings
import gc
from pathlib import Path

# --------------------------------------------------------------------------
# 0. GPU ACCELERATION INIT (RAPIDS cuML)
# --------------------------------------------------------------------------
# This intercepts standard scikit-learn imports and automatically
# proxies them to the GPU. This is the script equivalent of %load_ext cuml.accel
try:
    import cuml.accel
    cuml.accel.install()
    print("[0/6] NVIDIA cuML Acceleration ENABLED. Scikit-learn will run on GPU.")
except ImportError:
    print("[0/6] [!] cuML not found. Falling back to CPU. Ensure you are on a T4 GPU with RAPIDS installed.")

import numpy as np
import pandas as pd
import joblib  # Optimized serialization for models with large arrays

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

# Optional: The MLP Neural Net is kept here, but note that cuML does not
# accelerate MLPClassifier natively. It will run on the CPU.
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# --------------------------------------------------------------------------
# 1. DATA LOADING & MEMORY OPTIMIZATION
# --------------------------------------------------------------------------
def load_and_build_features(csv_path: str = "/content/d.csv", target_col: str = "diseases", drop_duplicates: bool = False):
    print(f"[1/6] Loading data from: {csv_path}")
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    if target_col not in df.columns:
        raise ValueError(f"Could not find target column '{target_col}'.")

    n_before = len(df)
    dup_count = df.duplicated().sum()
    print(f"      -> {n_before} rows loaded, {dup_count} duplicate rows detected.")

    if drop_duplicates and dup_count > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"      -> --drop_duplicates set: removed {dup_count} rows, {len(df)} remain.")

    y = df[target_col].astype(str).str.strip().values
    feature_cols = [c for c in df.columns if c != target_col]

    print("[2/6] Formatting matrix to dense float32 for GPU efficiency...")
    # CRITICAL MEMORY FIX: Force dataset into a dense float32 array immediately.
    # At 246k x 377, a float32 array is only ~372 MB. This stops Pandas from hogging RAM.
    X_df = df[feature_cols].fillna(0).astype(np.float32)
    feature_names = feature_cols
    X = X_df.values

    # Aggressively free up CPU RAM by deleting the original DataFrames
    del df
    del X_df
    gc.collect()

    print(f"      -> Matrix created: {X.shape[0]} samples, {X.shape[1]} features.")
    return X, y, feature_names

# --------------------------------------------------------------------------
# 2. CLASS-SAFE SPLITTING
# --------------------------------------------------------------------------
def stratified_split_with_full_coverage(X, y, test_size=0.2):
    print("[3/6] Building stratified train/test split...")
    counts = pd.Series(y).value_counts()
    rare_classes = counts[counts < 2].index.tolist()
    common_mask = ~pd.Series(y).isin(rare_classes).values

    X_common, y_common = X[common_mask], y[common_mask]
    X_rare, y_rare = X[~common_mask], y[~common_mask]

    X_train, X_test, y_train, y_test = train_test_split(
        X_common,
        y_common,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y_common,
    )

    if len(y_rare) > 0:
        X_train = np.vstack([X_train, X_rare])
        y_train = np.concatenate([y_train, y_rare])
        print(f"      -> {len(rare_classes)} rare disease(s) kept in TRAIN only.")

    return X_train, X_test, y_train, y_test, rare_classes

# --------------------------------------------------------------------------
# 3. GPU-ACCELERATED MODEL DEFINITIONS
# --------------------------------------------------------------------------
def get_models():
    """
    Returns a dict of {model_name: (estimator, needs_scaling)}
    """
    models = {
        "Decision Tree": (
            DecisionTreeClassifier(
                max_depth=15,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=RANDOM_STATE,
            ),
            False,
        ),
        "Random Forest": (
            # Handled directly by cuML proxy, offloading to the GPU VRAM
            RandomForestClassifier(
                n_estimators=100,
                max_depth=15,
                min_samples_split=10,
                min_samples_leaf=5,
                max_leaf_nodes=2000,
                n_jobs=-1,
                random_state=RANDOM_STATE,
                class_weight="balanced",
            ),
            False,
        ),
        "SVM (LinearSVC)": (
            LinearSVC(
                C=1.0,
                max_iter=2000,
                random_state=RANDOM_STATE,
                class_weight="balanced",
            ),
            True,
        ),
    }

    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = (
            XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                # CRITICAL: These parameters unlock native XGBoost CUDA acceleration
                tree_method="hist",
                device="cuda",
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
            ),
            False,
        )
    except ImportError:
        print("      [!] XGBoost not installed — skipping.")

    return models

# --------------------------------------------------------------------------
# 4. TRAINING & EVALUATION PIPELINE
# --------------------------------------------------------------------------
def train_evaluate_and_save(
    name, estimator, needs_scaling, X_train, X_test, y_train, y_test, scaler, out_dir
):
    print(f"\n----- {name} -----")
    Xtr, Xte = X_train, X_test
    if needs_scaling:
        Xtr = scaler.transform(X_train)
        Xte = scaler.transform(X_test)

    # Train
    t0 = time.time()
    estimator.fit(Xtr, y_train)
    train_time = time.time() - t0

    # Predict
    t0 = time.time()
    y_pred = estimator.predict(Xte)
    infer_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"Train time:        {train_time:.2f}s | Inference time: {infer_time:.2f}s")

    # SAVE THE MODEL (.pkl format via joblib for optimal matrix serialization)
    clean_name = name.replace(" ", "_").replace("(", "").replace(")", "").lower()
    model_path = out_dir / f"{clean_name}_model.pkl"
    joblib.dump(estimator, model_path)
    print(f"Model successfully saved to: {model_path}")

    metrics = {
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "train_time_s": round(train_time, 2),
    }
    return metrics, y_pred, estimator

# --------------------------------------------------------------------------
# 5. MAIN EXECUTION
# --------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(description="GPU Disease Prediction Pipeline.")
    parser.add_argument("--data", required=True, help="Path to dataset CSV.")
    parser.add_argument("--drop_duplicates", action="store_true", help="Drop fully-duplicate rows.")
    parser.add_argument("--out_dir", default="outputs", help="Directory for models/reports.")

    # Parse args (Replace with sys.argv checks if running natively in Colab cells)
    # args = parser.parse_args()
    # Hardcoded for immediate Colab execution below:
    class Args:
        data = "/content/d.csv"
        drop_duplicates = True
        out_dir = "/content/outputs"
    args = Args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    X, y, feature_names = load_and_build_features(args.data, drop_duplicates=args.drop_duplicates)
    X_train, X_test, y_train, y_test, _ = stratified_split_with_full_coverage(X, y)

    # Encode labels mathematically for GPU compatibility
    label_encoder = LabelEncoder().fit(y)
    y_train_enc = label_encoder.transform(y_train)
    y_test_enc = label_encoder.transform(y_test)

    # Save the label encoder to map predictions back to disease names later
    joblib.dump(label_encoder, out_dir / "label_encoder.pkl")

    scaler = StandardScaler(with_mean=False)
    scaler.fit(X_train)
    joblib.dump(scaler, out_dir / "standard_scaler.pkl")

    print("\n[4/6] Initializing GPU models...")
    models = get_models()

    print("\n[5/6] Training, evaluating, and serializing...")
    results = []
    predictions = {}
    best_accuracy = -1.0
    best_model_name = ""

    for name, (estimator, needs_scaling) in models.items():
        metrics, y_pred_enc, fitted_model = train_evaluate_and_save(
            name, estimator, needs_scaling, X_train, X_test, y_train_enc, y_test_enc, scaler, out_dir
        )
        results.append(metrics)
        predictions[name] = label_encoder.inverse_transform(y_pred_enc)

        if metrics["accuracy"] > best_accuracy:
            best_accuracy = metrics["accuracy"]
            best_model_name = name

        # Force garbage collection between models
        del estimator
        del fitted_model
        gc.collect()

    print("\n[6/6] Pipeline Complete.")
    results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
    print("\n================ MODEL COMPARISON ================")
    print(results_df.to_string(index=False))

    print(f"\nBest model by accuracy: {best_model_name}")
    report = classification_report(y_test, predictions[best_model_name], zero_division=0)
    report_path = out_dir / f"best_model_classification_report.txt"
    report_path.write_text(report)
    print(f"Full per-disease classification report saved -> {report_path}")

if __name__ == "__main__":
    main()

[0/6] NVIDIA cuML Acceleration ENABLED. Scikit-learn will run on GPU.
[1/6] Loading data from: /content/d.csv
      -> 246945 rows loaded, 57298 duplicate rows detected.
      -> --drop_duplicates set: removed 57298 rows, 189647 remain.
[2/6] Formatting matrix to dense float32 for GPU efficiency...
      -> Matrix created: 189647 samples, 377 features.
[3/6] Building stratified train/test split...
      -> 46 rare disease(s) kept in TRAIN only.

[4/6] Initializing GPU models...

[5/6] Training, evaluating, and serializing...

----- Decision Tree -----
Accuracy:          0.0775
Balanced Accuracy: 0.0249
Train time:        9.53s | Inference time: 0.08s
Model successfully saved to: /content/outputs/decision_tree_model.pkl

----- Random Forest -----
Accuracy:          0.6383
Balanced Accuracy: 0.6177
Train time:        34.79s | Inference time: 8.46s
Model successfully saved to: /content/outputs/random_forest_model.pkl

----- SVM (LinearSVC) -----
Accuracy:          0.8275
Balanced Accuracy